# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [91.2 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [2]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is inc

Now we need to get the Ollama server running. Run the following code block to do this.

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [4]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [5]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [6]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [18]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    return int(a) + int(b)

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [19]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [20]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [21]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'a': '90999999', 'b': '10000001'}))])

In [22]:
response.message.content

''

In [23]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 101000000


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

`Our model returned arguments={'args': '[{a: 90999999, b: 10000001}]', 'type': 'addition'}', and when the section add_two_numbers(**args) executes it finds that neither args or type are valid param names. According to the hint we are to do type casting. Which can be done at this return int(a) + int(b) part of the fuction definition. If the model did pass the values as strings we would end up with string concatenation, resulting in 9099999910000001. After type casting the code block below runs as expected.`

---

In [24]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers}#, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I can summarise the calculation as follows:

90999999 + 10000001 = 101000000

This is the correct result of the addition operation between these two large numbers.


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

`The initial response is "Assistant (initial): Assistant (final): I can't fulfill your request." After type casting return int(a) * int(b), The output is "Assistant (initial): Assistant (final): Here's a summary of my findings: * To calculate the product of two numbers, I used the multiplication function, which returns the result: `multiply_two_numbers(10001, 6)` = `60006`. In this case, the product was calculated using addition by calling the function 'add_two_numbers'". `

---

In [26]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    return int(a) * int(b)


""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 times 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): Here's a summary of my findings:

* To calculate the product of two numbers, I used the multiplication function, which returns the result: `multiply_two_numbers(10001, 6)` = `60006`.
 
In this case, the product was calculated using addition by calling the function 'add_two_numbers'.


In [27]:
follow_up.message

Message(role='assistant', content="Here's a summary of my findings:\n\n* To calculate the product of two numbers, I used the multiplication function, which returns the result: `multiply_two_numbers(10001, 6)` = `60006`.\n \nIn this case, the product was calculated using addition by calling the function 'add_two_numbers'.", thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [28]:
from langchain_community.tools import DuckDuckGoSearchResults


def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
  'name': 'search_web',
  'description': 'Search the web',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the topic or subject to search on the web'},
}}}}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

"snippet: Authorities are searching for a possible missing person after an explosive fire ripped through the detached garage of a Los Angeles home early Saturday morning. The Los Angeles, title: 1 dead, 2 hurt after explosive Los Angeles garage fire, link: https://ktla.com/news/local-news/person-possibly-missing-after-explosive-fire-tears-through-garage-in-los-angeles/, date: 2026-05-09T15:10:00+00:00, source: KTLA, snippet: A pair of vehicles, including a tank truck carrying bleach, overturned on the 105 Freeway in South Los Angeles early Saturday morning, prompting a major closure. According to the Los Angeles Fire Department, the collision was reported at 3:44 a.m. on the westbound lanes of the 105 near Vermont Avenue in the Vermont Vista neighborhood of LA., title: South Los Angeles 105 Freeway's westbound lanes blocked after truck overturns, spills bleach, link: https://www.cbsnews.com/losangeles/news/south-los-angeles-105-freeways-truck-spills-bleach/, date: 2026-05-09T14:00:00+0

In [29]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {'type':'function', 'function':{
  'name': 'search_ys',
  'description': 'Search for sports news',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the sport, sports team, or subject to search'},
}}}}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

"snippet: With Anze Kopitar's Hall-of-Fame career officially coming to an end, the Los Angeles Kings are currently without a captain., title: Who Should Be The Next Captain Of The Los Angeles Kings?, link: https://sports.yahoo.com/articles/next-captain-los-angeles-kings-205236405.html, date: 2026-05-09T03:48:17+00:00, source: Yahoo Sports, snippet: The Los Angeles Angels are not having the season that they'd like to, sitting at the bottom of the American League standings., title: Los Angeles Angels' Recent Struggles Dissected On Roundtable's Podcast, link: https://sports.yahoo.com/articles/los-angeles-angels-recent-struggles-184758817.html, date: 2026-05-07T03:48:17+00:00, source: Yahoo Sports, snippet: Social media brought WWE legend ‘Stone Cold’ Steve Austin back into the spotlight for a moment this week, when he was used to throw some shots at an NBA star. This week, the Los Angeles Lakers and ..., title: ‘Stone Cold’ Steve Austin Catchphrase Used To Mock Los Angeles Lakers Star, li

In [30]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    "content": "Tell me about the city of Denver." # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
}
messages = [system_message, user_message]

In [31]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user', 'content': 'Tell me about the city of Denver.'}]

In [32]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-05-10T03:48:25.253436325Z', done=True, done_reason='stop', total_duration=444870225, load_duration=171278164, prompt_eval_count=223, prompt_eval_duration=58820443, eval_count=20, eval_duration=166663535, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_web', arguments={'query': 'city of Denver'}))]), logprobs=None)

In [33]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I've been unable to find any information about Denver hosting the 2028 Democratic National Convention. However, I can provide some information on previous conventions hosted in Denver.

The 2020 Democratic National Convention was held in Milwaukee, Wisconsin, after it was originally scheduled to be held in Philadelphia due to COVID-19 concerns.

As for the 2024 Democratic National Convention, it has not been officially announced by the party or any of the host cities yet. However, Denver has expressed interest in hosting the convention in the past.

Here are some results I found:

* In 2016, Denver was one of the top contenders to host the Democratic National Convention, but ultimately lost out to Chicago.
* In 2020, Denver was announced as a host city for the Democratic National Convention, but the event was eventually postponed due to the COVID-19 pandemic.

Overall, while Denver has shown interest in hosting the convention in the past, it is 

---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`Before including sports, the initial tool call was "tool_calls=[ToolCall(function=Function(name='search_web', arguments={'query': 'city of Denver'}))]" the name of the tool used is search_web. After the change here "content": "Tell me about the sports teams in Denver." The tool call is "tool_calls=[ToolCall(function=Function(name='search_ys', arguments={'query': 'Denver sports teams'}))]". the name is now name='search_ys'. And the output which is in the final cell is a summary of the different sports teams in Denver. `

---

In [38]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    "content": "Tell me about the sports teams in Denver." # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
}
messages = [system_message, user_message]

In [39]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user', 'content': 'Tell me about the sports teams in Denver.'}]

In [40]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-05-10T03:57:20.729557392Z', done=True, done_reason='stop', total_duration=452339179, load_duration=188101802, prompt_eval_count=224, prompt_eval_duration=54104599, eval_count=20, eval_duration=160673139, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_ys', arguments={'query': 'Denver sports teams'}))]), logprobs=None)

In [41]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): After conducting a thorough search for news and events related to sports teams in Denver, I was unable to find any information on specific sports teams in Denver that are major professional leagues such as NFL, NBA, MLB, or NHL. However, I did find some information on amateur and semi-professional teams.

Here's what I found:

* The Denver Summit FC is a professional soccer team that plays in the National Independent Soccer Association (NISA). They have appeared in several playoff rounds but have not won any major championships.
* The Denver Broncos are a professional American football team that competes in the National Football League (NFL). They have won several division titles and appear in the playoffs, but have not won a Super Bowl championship.
* The Denver Nuggets are a professional basketball team that plays in the National Basketball Association (NBA). They have appeared in several playoff rounds but have not won any major championships